In [1]:
from ait.photonic_testing.photonic_testing import *
MODBUS_PORT='/dev/tty.usbserial-B003T6PZ'
LASER_ADDRESSES = {"1028": 5,
                   "1270": 6,
                   "yj1430": 4,
                   "hk1430": 3,
                   "1510": 1,
                   "2330": 2}

### Overview

See `ait/photonic_testing.py` for `Laser` and `LaserProperties` along with the specific limits of the individual laser diodes.


## Initialize all the diodes

Running this cell will create the `lasers` dictionary with a `Laser` for each laser.

In [2]:
names = tuple(LASER_ADDRESSES.keys())
lasers = {name: Laser(name, address=LASER_ADDRESSES[name], MODBUS_PORT=MODBUS_PORT) for name in names}
for l in lasers.values():
    l.startup()

Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.
Raw ID from register: 0x1113
🆔 Found device ID: 4371
📦 Loaded device configuration.


### Turn on a laser

In [5]:
for l in lasers.values(): l.shutdown()

In [3]:
lasers['2330'].set_current_as_percent(1, autooff=10)

Programming limits for 2330, Maiman driver: S/N 8226...
tec_running: True, interlocked: False, started: True
Setting current to 114.20000000000002 mA 
...current: 114.2 mA, output power: 2.7683000000000004 mW, wavelength: 2330.5072 nm (temp: 30.83 deg_C)


114.2

Look at the status of one of them. It seems that the TEC status isn't polling well, but the temp changes.

In [55]:
lasers['2330'].status()

Laser Properties Name: 2330
Raw ID from register: 0x1113
Device ID: 4371
Serial Number: 8226
State: 0xf7
 Operation started: True
 Current Set Internal: True
 Enable Internal: True
 External NTC Denied: True
 Interlock Denied: True
Current: 114.2
Current Min: 0.0
Current Max: 120.0
Protection Threshold: 119.0
Driver Max Current: 250.0
Voltage: 3.1
Frequency: 0.0
Duration: 10.0
Raw PCB temperature (signed): 0
PCB Temp: 0.0
Current Set Calibration: 100.0
Raw PCB temperature (signed): 0
PCB Temperature: 0.0
TEC PID: (20, 1000, 1000)
TEC Voltage: 0.8
TEC Current Limit: 1.2
TEC Current: 0.3
TEC Temperature Setpoint: 20.0
TEC Temperature: 20.14
TEC NTC Coefficient: 3988.0
TEC State: 0x16
 TEC started: True
 TEC Set Internal: True
 TEC Enable Internal: True
Interlock State: 0x2
 Interlock: True
 LD Overcurrent: False
 LD Overheat: False
 External NTC Interlock: False
 TEC Error: False
 TEC Self-heat: False


### Make sure it is all off.

In [5]:
for name in names: lasers[name].shutdown()

### Low-level access

In the event that direct device control is needed.

See `ait/maiman_modbus/utils/utils.py` for python constants of register names and `ait/maiman_modbus/config/modbus_config.yaml` for the register addresses.

Look at methods on `ModbusDevice` (`ait/maiman_modbus/device/modbus_device.py`) for functions.


```python
from maiman_modbus.communication import ModbusCommunication
import maiman_modbus.utils as maiman_regs
from maiman_modbus.device.modbus_device_model import ModbusDeviceModel
from maiman_modbus.config import DeviceConfig
from maiman_modbus.device.modbus_device import ModbusDevice

d = ModbusDevice(port=MODBUS_PORT, slave_address=modbus_address)
d.comm.send_command(d.model.get_register(STATE_OF_TEC_COMMAND), MODBUS_START_TEC_COMMAND_VALUE)

print(d.comm.receive_response(d.model.get_register(STATE_OF_TEC_COMMAND)))
```
